In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok", "nest-asyncio"])

In [ ]:
import os
import glob
import shutil
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score, silhouette_score, davies_bouldin_score
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import threading
import time
import urllib.request
import json

In [ ]:
dataset_path = "krishijal_master_dataset.csv"
input_matches = glob.glob("/kaggle/input/**/krishijal_master_dataset.csv", recursive=True)
if len(input_matches) > 0:
    dataset_path = input_matches[0]
elif os.path.exists("data/processed/krishijal_master_dataset.csv"):
    dataset_path = "data/processed/krishijal_master_dataset.csv"
elif os.path.exists("../data/processed/krishijal_master_dataset.csv"):
    dataset_path = "../data/processed/krishijal_master_dataset.csv"

os.makedirs("data/processed", exist_ok=True)
os.makedirs("models/saved_models", exist_ok=True)
if dataset_path != "data/processed/krishijal_master_dataset.csv" and os.path.exists(dataset_path):
    shutil.copy(dataset_path, "data/processed/krishijal_master_dataset.csv")

df = pd.read_csv(dataset_path)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop_duplicates()

In [ ]:
crop_categories = sorted(df["crop_type"].dropna().unique().tolist())
stage_categories = sorted(df["growth_stage"].dropna().unique().tolist())
crop_map = {name: idx for idx, name in enumerate(crop_categories)}
stage_map = {name: idx for idx, name in enumerate(stage_categories)}
df["crop_code"] = df["crop_type"].map(crop_map).fillna(0).astype(int)
df["stage_code"] = df["growth_stage"].map(stage_map).fillna(1).astype(int)

feature_cols = [
    "temperature", "temp_max", "temp_min", "rainfall", "humidity",
    "effective_soil_wetness", "solar_radiation", "soil_ph", "soil_organic_carbon",
    "ndvi", "et0", "etc", "crop_code", "stage_code"
]
X = df[feature_cols]
y_class = df["stress_level_code"]
y_reg = df["irrigation_priority_index"]

In [ ]:
X_train, X_test, y_train_class, y_test_class, y_train_reg, y_test_reg = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_cols)

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x=df["stress_level"], hue=df["stress_level"], order=["Low", "Moderate", "High", "Severe"], palette="viridis", legend=False)
plt.title("Crop Water Stress Level Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df[feature_cols[:8]].corr(), annot=True, cmap="Blues", fmt=".2f")
plt.title("Correlation Heatmap of Key Agricultural Features")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["irrigation_priority_index"], bins=30, kde=True, color="teal")
plt.title("Distribution of Irrigation Priority Index")
plt.show()

In [ ]:
clf_logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled, y_train_class)
clf_dt = DecisionTreeClassifier(max_depth=8, random_state=42).fit(X_train_scaled, y_train_class)
clf_knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train_class)
clf_nb = GaussianNB().fit(X_train_scaled, y_train_class)
clf_rf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42).fit(X_train_scaled, y_train_class)

In [ ]:
models_clf = {
    "Logistic Regression": clf_logreg,
    "Decision Tree": clf_dt,
    "K-Nearest Neighbors": clf_knn,
    "Gaussian Naive Bayes": clf_nb,
    "Random Forest": clf_rf
}
clf_metrics = []
for name, m in models_clf.items():
    pred = m.predict(X_test_scaled)
    acc = accuracy_score(y_test_class, pred)
    prec = precision_score(y_test_class, pred, average="weighted", zero_division=0)
    rec = recall_score(y_test_class, pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test_class, pred, average="weighted", zero_division=0)
    clf_metrics.append({"Model": name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1})
df_clf_results = pd.DataFrame(clf_metrics)
df_clf_results

In [ ]:
cm = confusion_matrix(y_test_class, clf_rf.predict(X_test_scaled))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=["Low", "Mod", "High", "Sev"], yticklabels=["Low", "Mod", "High", "Sev"])
plt.title("Random Forest Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.show()

In [ ]:
reg_lin = LinearRegression().fit(X_train_scaled, y_train_reg)
reg_dt = DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_train_scaled, y_train_reg)
reg_poly = Pipeline([
    ("poly", PolynomialFeatures(degree=2)),
    ("linear", LinearRegression())
]).fit(X_train_scaled, y_train_reg)
reg_rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42).fit(X_train_scaled, y_train_reg)

In [ ]:
models_reg = {
    "Linear Regression": reg_lin,
    "Polynomial Regression (Deg 2)": reg_poly,
    "Decision Tree Regressor": reg_dt,
    "Random Forest Regressor": reg_rf
}
reg_metrics = []
for name, m in models_reg.items():
    pred = m.predict(X_test_scaled)
    mae = mean_absolute_error(y_test_reg, pred)
    mse = mean_squared_error(y_test_reg, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_reg, pred)
    reg_metrics.append({"Model": name, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2 Score": r2})
df_reg_results = pd.DataFrame(reg_metrics)
df_reg_results

In [ ]:
param_grid_rf = {
    "n_estimators": [50, 100],
    "max_depth": [8, 12]
}
grid_search_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=3, scoring="accuracy", n_jobs=-1)
grid_search_rf.fit(X_train_scaled, y_train_class)
best_tuned_rf = grid_search_rf.best_estimator_
grid_search_rf.best_params_

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_train_scaled)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train_scaled)
sil = silhouette_score(X_train_scaled, kmeans.labels_, sample_size=1000, random_state=42)
db = davies_bouldin_score(X_train_scaled, kmeans.labels_)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans.labels_, cmap="tab10", alpha=0.6, s=25)
plt.title("K-Means (k=4) Clusters via 2D PCA")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.colorbar(scatter, label="Cluster ID")
plt.show()

In [ ]:
os.makedirs("models/saved_models", exist_ok=True)
os.makedirs("models", exist_ok=True)
if os.path.exists("/kaggle/working"):
    os.makedirs("/kaggle/working/models/saved_models", exist_ok=True)
    os.makedirs("/kaggle/working/models", exist_ok=True)

joblib.dump(scaler, "models/saved_models/feature_scaler.pkl")
joblib.dump(best_tuned_rf, "models/saved_models/crop_stress_classifier.pkl")
joblib.dump(reg_rf, "models/saved_models/irrigation_priority_regressor.pkl")
joblib.dump(clf_dt, "models/saved_models/crop_stress_decision_tree.pkl")
joblib.dump(clf_logreg, "models/saved_models/crop_stress_logistic_regression.pkl")
joblib.dump(reg_dt, "models/saved_models/irrigation_decision_tree.pkl")
joblib.dump(reg_lin, "models/saved_models/irrigation_linear_regression.pkl")

for f in os.listdir("models/saved_models"):
    if f.endswith(".pkl"):
        shutil.copy(f"models/saved_models/{f}", f"models/{f}")
        if os.path.exists("/kaggle/working"):
            shutil.copy(f"models/saved_models/{f}", f"/kaggle/working/models/{f}")
            shutil.copy(f"models/saved_models/{f}", f"/kaggle/working/{f}")

In [ ]:
if not os.path.exists("main.py") and os.path.exists("SDC_KAGGLE/main.py"):
    shutil.copy("SDC_KAGGLE/main.py", "main.py")

In [ ]:
nest_asyncio.apply()

token = os.environ.get("NGROK_AUTHTOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    except Exception:
        token = input("Enter Ngrok AuthToken: ")

if token:
    ngrok.set_auth_token(token.strip())

ngrok.kill()
tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url

server_thread = threading.Thread(target=uvicorn.run, args=("main:app",), kwargs={"host": "0.0.0.0", "port": 8000, "reload": False}, daemon=True)
server_thread.start()

print(public_url)
print(public_url + "/docs")
print(public_url + "/health")

In [ ]:
time.sleep(2)
req = urllib.request.Request("http://127.0.0.1:8000/health")
with urllib.request.urlopen(req) as response:
    health_status = json.loads(response.read().decode())
health_status

In [ ]:
sample_payload = {
    "temperature": 34.0,
    "rainfall": 0.0,
    "humidity": 45.0,
    "effective_soil_wetness": 0.28,
    "crop_type": "Cotton",
    "growth_stage": "Reproductive"
}
req = urllib.request.Request(
    "http://127.0.0.1:8000/predict",
    data=json.dumps(sample_payload).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)
with urllib.request.urlopen(req) as response:
    prediction_result = json.loads(response.read().decode())
prediction_result